# Infidelidade da porta CPHASE com $N$ sítios cross-Kerr — Fig. 3 de Brod & Combes

Este notebook explica, expressão por expressão, tudo o que entra na figura gerada por
`plot_infidelidade.py` (dados calculados por `calcular_infidelidade.py`): a reprodução da
**Fig. 3** de Brod & Combes, *"A passive CPHASE gate via cross-Kerr nonlinearities"*
(arXiv:1604.04278), para dois fótons **contrapropagantes** que atravessam **$N$ sítios idênticos**,
com um deslocador de fase $e^{i\omega L}$ entre sítios vizinhos.

* **Fig. 3(a)**: fidelidade média $F_{\rm avg}(\phi=\pi)$ em função da largura do pacote $\sigma$, para vários $N$;
* **Fig. 3(b)**: infidelidade mínima $1-F_{\max}$ e largura ótima $\sigma_{\max}$ em função de $N$ (log-log),
  com ajustes de lei de potência.

**Roteiro**

| Seção | O que é explicado | Onde aparece no código |
|---|---|---|
| 1 | Parâmetros físicos | `chi, gamma, Delta, omega0, phi, L_values, N_values` |
| 2 | Blocos básicos: $\Gamma$, $t$, $a$, $S_1$, $\xi$ | `Gamma`, `t`, `a_ph`, `S1`, `xi_in` |
| 3 | Matriz S de dois fótons e $N$ sítios: $\chi K(E)$, soma em $j$ | `chiK`, `S2_conn` |
| 4 | Overlap $\mathrm{Ov}=1+G$ e a fatoração em convoluções | `_conv_terms`, `compute_G`, `overlap` |
| 5 | Fidelidade média $F_{\rm avg}(\phi)$ | `F_avg`, `fidelity` |
| 6 | Grade numérica de integração | `grade` |
| 7 | Validações: força bruta 3D e unitaridade | `brute_force_G`, `norm_defect` |
| 8 | Otimização em $\sigma$: $\sigma_{\max}$ e $F_{\max}$ | `optimize_sigma` |
| 9 | Ajustes de lei de potência | `fit_power_law` (em `plot_infidelidade.py`) |
| 10 | Verificação: recalcular e comparar com os `.npz` | — |
| 11 | Limite analítico $\sigma\to0$ | — |
| 12 | O gráfico, elemento por elemento | `plot_infidelidade.py` |

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import simpson
from scipy.optimize import minimize_scalar
from scipy.signal import fftconvolve

---
## 1. Parâmetros físicos

Todos os $N$ sítios são idênticos: cada um é um emissor/ressonador com

* **largura** (taxa de decaimento) $\gamma$,
* **dessintonia** $\Delta$,
* **não linearidade** cross-Kerr $\chi$.

Entre dois sítios consecutivos há um **deslocador de fase** que multiplica a amplitude de um fóton de
frequência $\omega$ por $e^{i\omega L}$ (em unidades em que a velocidade de propagação vale 1, $L$ é a
distância entre sítios). Com $L=0$ recupera-se exatamente o modelo do artigo.

O pacote de entrada é gaussiano, centrado em

$$
\omega_c=\Delta+\omega_0 ,
$$

com largura espectral $\sigma$ (eixo $x$ da Fig. 3(a)). A porta alvo é a CPHASE com $\phi=\pi$ (CZ).

Valores usados na figura: $\chi=1000$ (praticamente o limite saturado $\chi\to\infty$ do artigo, ver seção 3),
$\gamma=1$, $\Delta=0$, $\omega_0=0$, $N=1,\dots,50$ e $L=0.2$ (o artigo corresponde a $L=0$).

In [ ]:
# Parâmetros físicos (mesmos valores de calcular_infidelidade.py)
chi = 1000.0          # não linearidade (np.inf = limite saturado do artigo)
gamma = 1.0           # taxa de decaimento (igual em todos os sítios)
Delta = 0.0           # dessintonia dos sítios
omega0 = 0.0          # centro do pacote relativo a Delta
phi = np.pi           # fase da porta (CPHASE -> CZ)
L = 0.2               # deslocamento de fase entre sítios

omega_c = Delta + omega0   # frequência central efetiva

# Parâmetros numéricos (seções 6 e 8)
k_sigma = 9.0         # meia-janela de integração em unidades de sigma
pts_per_width = 14    # pontos por menor escala do integrando
n_min, n_max = 2001, 800001
sigma_lo, sigma_hi, n_sigma = 1e-3, 10.0, 121   # grade grossa da otimização em sigma

---
## 2. Blocos básicos

### 2.1 Denominador de Breit–Wigner $\Gamma(\omega)$

$$
\boxed{\;\Gamma(\omega)=\frac{\gamma}{2}+i\,(\Delta-\omega)\;}
$$

É o denominador da resposta linear de um sítio; anula-se no polo $\omega=\Delta-i\gamma/2$ (lorentziana
de largura $\gamma$). No código: `Gamma(w)`.

### 2.2 Fase de fóton único de um sítio $t(\omega)$

Um fóton sozinho atravessa o sítio sem perda, ganhando apenas uma fase:

$$
\boxed{\;t(\omega)=\frac{\Gamma^{*}(\omega)}{\Gamma(\omega)}\;},\qquad |t(\omega)|=1 .
$$

Perto da ressonância a fase de $t$ gira rápido: $\arg t$ varia $\approx 4/\gamma$ rad por unidade de
$\omega$ em $\omega=\Delta$. Isso define uma das escalas da grade numérica (seção 6).

### 2.3 Fase "sítio + deslocador" $a(\omega)$

$$
\boxed{\;a(\omega)=t(\omega)\,e^{i\omega L}\;}
$$

é a amplitude acumulada por um fóton ao passar por um sítio **e** por um deslocador de fase. No código:
`a_ph(w, L)`.

### 2.4 Amplitude de fóton único através dos $N$ sítios $S_1(\omega)$

Atravessando os $N$ sítios e os $N-1$ deslocadores entre eles:

$$
\boxed{\;S_1(\omega)=e^{i(N-1)\omega L}\,t(\omega)^{N}\;}
$$

O artigo escreve $-[\Gamma^*/\Gamma]^N$; o sinal é uma fase global de fóton único e desaparece quando a
deformação de fóton único é removida (seção 4). No código: `S1(w, N, L)`.

### 2.5 Pacote de onda gaussiano $\xi(\omega)$ (Eq. 4 do artigo)

$$
\boxed{\;\xi(\omega)=\left(2\pi\sigma^{2}\right)^{-1/4}
\exp\!\left[-\frac{(\omega-\omega_c)^{2}}{4\sigma^{2}}\right]\;},
\qquad \int|\xi(\omega)|^{2}\,d\omega=1 .
$$

$|\xi|^2$ é uma gaussiana de desvio padrão $\sigma$. Os dois fótons (a e b) entram com o mesmo perfil,
de modo que o estado de entrada tem amplitude $\xi(\nu_a)\,\xi(\nu_b)$.

In [ ]:
def Gamma(w):
    return gamma / 2 + 1j * (Delta - w)


def xi_in(w, sigma):
    """Pacote gaussiano normalizado, Eq. (4)."""
    return (2 * np.pi * sigma ** 2) ** -0.25 * np.exp(-(w - omega_c) ** 2 / (4 * sigma ** 2))


def t(w):
    """Fase de fóton único de um sítio, Γ*/Γ (|t| = 1)."""
    return np.conj(Gamma(w)) / Gamma(w)


def a_ph(w, L):
    """Fase 'sítio + deslocador': a(ω) = t(ω) e^{iωL}."""
    return t(w) * np.exp(1j * w * L)


def S1(w, N, L):
    """Amplitude de fóton único através dos N sítios."""
    return np.exp(1j * (N - 1) * w * L) * t(w) ** N


# --- checagens rápidas das propriedades enunciadas acima ---
w = np.linspace(-20, 20, 4001)
print("|t| = 1          :", np.allclose(np.abs(t(w)), 1.0))
print("|S1| = 1 (N=7)   :", np.allclose(np.abs(S1(w, 7, L)), 1.0))
print("∫|ξ|² dω (σ=0.7) :", simpson(np.abs(xi_in(w, 0.7)) ** 2, x=w))

---
## 3. Matriz S de dois fótons e $N$ sítios

Os fótons entram com frequências $(\nu_a,\nu_b)$ e saem com $(\omega_a,\omega_b)$; $a$ e $b$ se propagam em
sentidos opostos. A matriz S tem uma parte **produto** (fótons independentes) e uma parte **conectada**
(interação), que conserva só a energia total:

$$
\boxed{\;
\begin{aligned}
S(\omega_a,\omega_b;\nu_a,\nu_b)=\;&S_1(\omega_a)\,S_1(\omega_b)\,\delta(\omega_a-\nu_a)\,\delta(\omega_b-\nu_b)\\
&-\frac{i\gamma^{2}}{\pi}\,\frac{\chi K(E)\;\delta(\omega_a+\omega_b-\nu_a-\nu_b)}
{\Gamma(\nu_a)\Gamma(\nu_b)\Gamma(\omega_a)\Gamma(\omega_b)}
\sum_{j=1}^{N}A^{N-j}B^{\,j-1}
\end{aligned}\;}
$$

com a **energia total** $E=\nu_a+\nu_b=\omega_a+\omega_b$ e

$$
A=a(\omega_a)\,a(\nu_b),\qquad B=a(\omega_b)\,a(\nu_a).
$$

### 3.1 Fator de saturação $\chi K(E)$

$$
\boxed{\;K(E)=\left[1+\frac{2i\chi}{\gamma+i(2\Delta-E)}\right]^{-1}
\quad\Longrightarrow\quad
\chi K(E)=\frac{\chi\,D}{D+2i\chi},\qquad D=\gamma+i(2\Delta-E)\;}
$$

Note que $D=\Gamma(\omega_a)+\Gamma(\omega_b)$, que só depende de $E$: o fator de interação **não depende
de como a energia é repartida** entre os fótons — é isso que permite fatorizar a integral (seção 4).

* $\chi\to0$: $\chi K\to0$, sem interação;
* $\chi\to\infty$: $\chi K\to D/(2i)$ — o produto **satura** (bloqueio: dois fótons não ocupam o mesmo
  sítio). $\chi=1000\gg\gamma$ já está praticamente nesse limite. No código, `chiK(E, np.inf)` usa a forma
  saturada exata.

### 3.2 A soma sobre os sítios $\sum_j A^{N-j}B^{j-1}$

O termo $j$ corresponde à interação ocorrer no **sítio $j$**. Como os fótons são contrapropagantes, cada
um encontra os sítios em ordem oposta:

| Fóton | Antes do sítio $j$ | Depois do sítio $j$ |
|---|---|---|
| $a$ (entra pela esquerda) | atravessa $j-1$ sítios com $\nu_a$: $a(\nu_a)^{j-1}$ | atravessa $N-j$ sítios com $\omega_a$: $a(\omega_a)^{N-j}$ |
| $b$ (entra pela direita) | atravessa $N-j$ sítios com $\nu_b$: $a(\nu_b)^{N-j}$ | atravessa $j-1$ sítios com $\omega_b$: $a(\omega_b)^{j-1}$ |

Juntando: $[a(\omega_a)a(\nu_b)]^{N-j}\,[a(\omega_b)a(\nu_a)]^{j-1}=A^{N-j}B^{j-1}$. Cada sítio
"espectador" contribui só com a fase linear $a$.

É uma série geométrica, com forma fechada

$$
\sum_{j=1}^{N}A^{N-j}B^{j-1}=\frac{A^{N}-B^{N}}{A-B}\qquad(\to N A^{N-1}\ \text{se}\ A\to B),
$$

usada em `S2_conn` (só para a checagem por força bruta da seção 7). O cálculo de produção mantém a soma
explícita, porque cada termo fatoriza (seção 4).

In [ ]:
def chiK(E, chi_=None):
    """χ K(E). Com χ = ∞ usa o limite saturado (γ + i(2Δ-E)) / (2i)."""
    chi_ = chi if chi_ is None else chi_
    D = gamma + 1j * (2 * Delta - E)
    if np.isinf(chi_):
        return D / 2j
    return chi_ * D / (D + 2j * chi_)


def S2_conn(wa, wb, va, vb, N, L, chi_=None):
    """Parte conectada de <ωa ωb|S|νa νb>, sem a delta (série geométrica em forma fechada)."""
    A = a_ph(wa, L) * a_ph(vb, L)
    B = a_ph(wb, L) * a_ph(va, L)
    diff = A - B
    close = np.abs(diff) < 1e-8
    geo = np.where(close, N * A ** (N - 1),
                   (A ** N - B ** N) / np.where(close, 1.0, diff))
    E = va + vb
    return (-1j * gamma ** 2 / np.pi) * chiK(E, chi_) * geo / (
        Gamma(va) * Gamma(vb) * Gamma(wa) * Gamma(wb))


# χK satura: com χ = 1000 já está a ~1e-3 do limite χ → ∞
E = np.linspace(-3, 3, 7)
print("max |χK(1000) - χK(∞)| / |χK(∞)| =", np.max(np.abs(chiK(E) - chiK(E, np.inf)) / np.abs(chiK(E, np.inf))))

# forma fechada da série geométrica vs soma explícita
A_, B_, N_ = 0.3 + 0.9j, -0.5 + 0.2j, 7
print("série geométrica ok:", np.isclose(sum(A_ ** (N_ - j) * B_ ** (j - 1) for j in range(1, N_ + 1)),
                                         (A_ ** N_ - B_ ** N_) / (A_ - B_)))

---
## 4. Overlap $\mathrm{Ov}=1+G$ (Eq. A3)

A fidelidade depende da **amplitude de sobreposição** entre o estado de entrada e o de saída, depois de
remover a evolução de fóton único (uma fase local, corrigível por unitárias de um qubit), isto é, aplicando
$S_1^\dagger(\omega_a)S_1^\dagger(\omega_b)$:

$$
\mathrm{Ov}=\langle\xi\xi|\,S_1^\dagger S_1^\dagger\,S\,|\xi\xi\rangle .
$$

* **Parte produto**: $S_1^*S_1=|S_1|^2=1$ e $\xi$ é normalizada, logo ela contribui exatamente $1$.
* **Parte conectada**: contribui com a integral $G$.

$$
\boxed{\;\mathrm{Ov}=1+G\;}
$$

$$
\boxed{\;
G=\int d\omega_a\,d\omega_b\,d\nu_a\,d\nu_b\;
\xi(\omega_a)\xi(\omega_b)\,S_1^{*}(\omega_a)S_1^{*}(\omega_b)\;
S_{\rm conn}(\omega_a,\omega_b;\nu_a,\nu_b)\;\xi(\nu_a)\xi(\nu_b)\;}
$$

($\xi$ é real, então $\xi^*=\xi$.) Uma porta perfeita teria $\mathrm{Ov}=-1$ (fase condicional $\pi$, sem
perdas); $|\mathrm{Ov}|<1$ significa **vazamento** para fora do subespaço computacional (os fótons saem com o
espectro deformado).

### 4.1 Fatoração em convoluções

A delta elimina $\omega_b=E-\omega_a$ e deixa uma integral tripla. Mas, para cada $j$,

$$
A^{N-j}B^{j-1}=\big[a^{N-j}(\omega_a)\,a^{j-1}(\omega_b)\big]\;\big[a^{j-1}(\nu_a)\,a^{N-j}(\nu_b)\big],
$$

e $\chi K$ depende só de $E$. Todos os demais fatores dependem de **uma única** frequência. Definindo

$$
b_{\rm out}(\omega)=\frac{\xi(\omega)\,S_1^{*}(\omega)}{\Gamma(\omega)},\qquad
b_{\rm in}(\nu)=\frac{\xi(\nu)}{\Gamma(\nu)},
$$

$$
C^{\rm out}_j(E)=\int d\omega\;\big[b_{\rm out}a^{N-j}\big](\omega)\,\big[b_{\rm out}a^{j-1}\big](E-\omega),\qquad
C^{\rm in}_j(E)=\int d\nu\;\big[b_{\rm in}a^{j-1}\big](\nu)\,\big[b_{\rm in}a^{N-j}\big](E-\nu),
$$

a integral vira uma soma de integrais **unidimensionais** em $E$:

$$
\boxed{\;G=-\frac{i\gamma^{2}}{\pi}\int dE\;\chi K(E)\sum_{j=1}^{N}C^{\rm out}_j(E)\,C^{\rm in}_j(E)\;}
$$

As $2N$ convoluções são feitas por FFT (`fftconvolve(f, g) * h`, com $h$ o passo da grade, que converte a
soma discreta em integral) e a integral em $E$ por Simpson. O custo cai de $O(n^3)$ para $O(N\,n\log n)$.
O eixo $E$ vai de $2\omega_{\min}$ a $2\omega_{\max}$ (soma de duas frequências da grade).

In [ ]:
def grade(sigma, N, L, pts=None, ks=None):
    """Grade uniforme de frequências (ver seção 6)."""
    pts = pts_per_width if pts is None else pts
    ks = k_sigma if ks is None else ks
    half = ks * sigma
    scales = [sigma, gamma / (2 * N)]
    if L != 0:
        scales.append(1.0 / (N * abs(L)))
    n = int(2 * half / (min(scales) / pts)) + 1
    n = min(max(n, n_min), n_max) | 1       # ímpar -> Simpson com nº par de intervalos
    w = np.linspace(omega_c - half, omega_c + half, n)
    return w, w[1] - w[0]


def _conv_terms(w, h, sigma, N, L):
    """Devolve (E, [C_out_j(E)], [C_in_j(E)])."""
    xi, G = xi_in(w, sigma), Gamma(w)
    a = a_ph(w, L)
    base_out = xi * np.conj(S1(w, N, L)) / G    # b_out
    base_in = xi / G                            # b_in
    E = 2 * w[0] + h * np.arange(2 * len(w) - 1)
    conv_out, conv_in = [], []
    for j in range(1, N + 1):
        f, g = a ** (N - j), a ** (j - 1)
        conv_out.append(fftconvolve(base_out * f, base_out * g) * h)
        conv_in.append(fftconvolve(base_in * g, base_in * f) * h)
    return E, conv_out, conv_in


def compute_G(sigma, N, L, chi_=None, pts=None, ks=None):
    """G = Ov - 1 (seção 4.1)."""
    w, h = grade(sigma, N, L, pts, ks)
    E, co, ci = _conv_terms(w, h, sigma, N, L)
    soma = sum(o * i for o, i in zip(co, ci))
    return (-1j * gamma ** 2 / np.pi) * simpson(chiK(E, chi_) * soma, x=E)


def overlap(sigma, N, L, chi_=None, **kw):
    return 1.0 + compute_G(sigma, N, L, chi_, **kw)


for N in (1, 2, 5, 20):
    Ov = overlap(0.1, N, L)
    print(f"N={N:2d}  σ=0.1  Ov = {Ov:.4f}   |Ov| = {abs(Ov):.4f}")

---
## 5. Fidelidade média $F_{\rm avg}(\phi)$ (Eq. A2)

No subespaço computacional de dois qubits ($d=4$) a operação efetiva é $M=\mathrm{diag}(1,1,1,\mathrm{Ov})$:
só em $|11\rangle$ os dois fótons interagem. A porta alvo é $U=\mathrm{diag}(1,1,1,e^{-i\phi})$. A fidelidade
média sobre estados de Haar, tratando o vazamento corretamente, é

$$
F_{\rm avg}=\frac{\big|\mathrm{Tr}(U^\dagger M)\big|^{2}+\mathrm{Tr}(M^\dagger M)}{d\,(d+1)},
\qquad \mathrm{Tr}(U^\dagger M)=3+e^{i\phi}\mathrm{Ov},\quad \mathrm{Tr}(M^\dagger M)=3+|\mathrm{Ov}|^{2},
$$

o que dá

$$
\boxed{\;F_{\rm avg}(\phi)=\frac{6+3\,\mathrm{Re}\!\left(e^{i\phi}\,\mathrm{Ov}\right)+|\mathrm{Ov}|^{2}}{10}\;}
$$

**É esta a grandeza do eixo $y$ da Fig. 3(a)**, com $\phi=\pi$:
$F_{\rm avg}(\pi)=\big[6-3\,\mathrm{Re}\,\mathrm{Ov}+|\mathrm{Ov}|^{2}\big]/10$.

| Situação | $\mathrm{Ov}$ | $F_{\rm avg}(\pi)$ |
|---|---|---|
| sem interação ($\chi=0$ ou $\sigma\to0$) | $1$ | $(6-3+1)/10=\mathbf{0.4}$ |
| porta perfeita | $-1$ | $(6+3+1)/10=\mathbf{1}$ |

In [ ]:
def F_avg(Ov, phi_=phi):
    """Fidelidade média de Haar (Eq. A2)."""
    return (6 + 3 * np.real(np.exp(1j * phi_) * Ov) + np.abs(Ov) ** 2) / 10.0


def fidelity(sigma, N, L, **kw):
    return F_avg(overlap(sigma, N, L, **kw))


print("F_avg(π | Ov= 1) =", F_avg(1.0))
print("F_avg(π | Ov=-1) =", F_avg(-1.0))
Ov0 = overlap(0.3, 5, 0.4, chi_=0.0)
print("χ = 0  ->  Ov =", Ov0, "  F_avg(π) =", F_avg(Ov0))

---
## 6. Grade numérica de integração

Grade uniforme $\omega\in[\omega_c-k_\sigma\sigma,\;\omega_c+k_\sigma\sigma]$, com $k_\sigma=9$: as quatro gaussianas
confinam o integrando nessa janela.

O espaçamento deve resolver a **menor escala** do integrando, com `pts_per_width` $=14$ pontos por escala:

$$
n=\frac{2k_\sigma\sigma}{\ell_{\min}/14}+1,\qquad
\ell_{\min}=\min\!\left(\sigma,\;\frac{\gamma}{2N},\;\frac{1}{N|L|}\right),\qquad n\in[2001,\;800001],\ n\ \text{ímpar}
$$

* $\sigma$: largura do pacote;
* $\gamma/(2N)$: a fase $t^N$ gira $\approx 4N/\gamma$ rad por unidade de $\omega$ na ressonância;
* $1/(N|L|)$: as fases $e^{i\omega L}$ acumuladas em até $N$ sítios (só se $L\neq0$).

$n$ ímpar $\Rightarrow$ número par de intervalos, exigido pela regra de Simpson.

In [ ]:
for N, s in [(1, 1.0), (4, 0.1), (20, 0.05), (50, 0.02)]:
    wg, h = grade(s, N, L)
    print(f"N={N:2d}  σ={s:5.2f}  n={len(wg):7d}  janela=±{k_sigma*s:5.2f}  passo h={h:.2e}")

---
## 7. Validações

### 7.1 Convolução vs. força bruta 3D

A fatoração da seção 4.1 é conferida contra a integral tripla direta (trapézio em $\omega_a,\nu_a,\nu_b$,
com $\omega_b=\nu_a+\nu_b-\omega_a$) usando a matriz S pontual `S2_conn` (série geométrica em forma fechada).

### 7.2 Unitaridade

Se $S$ é unitária, o estado de saída tem norma 1. Escrevendo o estado de saída (já com $S_1^\dagger S_1^\dagger$)
como $\xi\xi+\tilde\psi$, com $\tilde\psi$ a parte conectada,

$$
\|\xi\xi+\tilde\psi\|^2-1 \;=\; 2\,\mathrm{Re}\,G+\iint d\omega_a\,d\omega_b\,|\tilde\psi(\omega_a,\omega_b)|^2 \;=\; 0 .
$$

$\tilde\psi$ decai só como $1/u^2$ (com $\omega_{a,b}=E/2\pm u$), muito além da janela gaussiana; por isso a
integral em $u$ é feita até $\pm\infty$ com a troca $u=(\gamma/2)\tan\theta$. Este teste é sensível a
qualquer erro de prefator, sinal ou fase na matriz S.

In [ ]:
def brute_force_G(sigma, N, L, n=161, ks=6.0):
    """G por integração 3D direta com S2_conn pontual."""
    x = np.linspace(omega_c - ks * sigma, omega_c + ks * sigma, n)
    h = x[1] - x[0]
    wa, va, vb = np.meshgrid(x, x, x, indexing='ij', sparse=True)
    wb = va + vb - wa
    integ = (xi_in(wa, sigma) * xi_in(wb, sigma) * xi_in(va, sigma) * xi_in(vb, sigma)
             * np.conj(S1(wa, N, L) * S1(wb, N, L)) * S2_conn(wa, wb, va, vb, N, L))
    return integ.sum() * h ** 3


def norm_defect(sigma, N, L, n_theta=8001):
    """2 Re G + ∫∫|ψ̃|²: deve ser 0 se S for unitária."""
    w, h = grade(sigma, N, L)
    E, _, ci = _conv_terms(w, h, sigma, N, L)
    s = max(1, len(E) // 600)
    E, ci = E[::s], [c[::s] for c in ci]
    theta = np.linspace(-np.pi / 2, np.pi / 2, n_theta)[1:-1]
    u = gamma / 2 * np.tan(theta)
    jac = gamma / 2 / np.cos(theta) ** 2
    inner = np.empty(len(E))
    for k, Ek in enumerate(E):
        wa, wb = Ek / 2 + u, Ek / 2 - u
        aa, ab = a_ph(wa, L), a_ph(wb, L)
        amp = sum(aa ** (N - j) * ab ** (j - 1) * ci[j - 1][k] for j in range(1, N + 1))
        psi = (gamma ** 2 / np.pi) * chiK(Ek) * amp / (Gamma(wa) * Gamma(wb))
        inner[k] = simpson(np.abs(psi) ** 2 * jac, x=theta)
    return 2 * np.real(compute_G(sigma, N, L)) + simpson(inner, x=E)


for N, L_, s in [(2, 0.0, 0.3), (3, 0.4, 0.3)]:
    Gc, Gb = compute_G(s, N, L_), brute_force_G(s, N, L_)
    print(f"força bruta (N={N}, L={L_:g}, σ={s:g}):  G_conv={Gc:.6f}  G_3D={Gb:.6f}  |ΔG|={abs(Gc - Gb):.1e}")

for N, L_, s in [(2, 0.4, 0.2), (6, 0.4, 0.1)]:
    print(f"unitaridade (N={N}, L={L_:g}, σ={s:g}):  defeito = {norm_defect(s, N, L_):.1e}")

---
## 8. Otimização em $\sigma$: $\sigma_{\max}$ e $F_{\max}$

Para cada $N$, a Fig. 3(b) usa o **melhor** pacote:

$$
\boxed{\;F_{\max}(N)=\max_{\sigma}F_{\rm avg}(\pi;\sigma,N),\qquad
\sigma_{\max}(N)=\arg\max_{\sigma}F_{\rm avg}(\pi;\sigma,N)\;}
$$

Procedimento (`optimize_sigma`):

1. grade logarítmica grossa $\sigma\in[10^{-3},10]$ com 121 pontos ($\sim40$ por década);
2. refino com o método de Brent limitado (`minimize_scalar(method='bounded')`) em $\log_{10}\sigma$, entre os
   dois vizinhos do máximo da grade, com tolerância $10^{-7}$ em $\log_{10}\sigma$.

Se o máximo cair na borda da grade, um aviso é impresso. A Fig. 3(a) não usa a otimização: ela mostra
$F_{\rm avg}(\pi)$ numa grade fina `np.logspace(-2, 0, 301)` para $N\in\{1,2,3,4,5,10,20,50\}$.

In [ ]:
def optimize_sigma(N, L, **kw):
    sig = np.logspace(np.log10(sigma_lo), np.log10(sigma_hi), n_sigma)
    F = np.array([fidelity(s, N, L, **kw) for s in sig])
    k = int(np.argmax(F))
    lo, hi = np.log10(sig[max(k - 1, 0)]), np.log10(sig[min(k + 1, len(sig) - 1)])
    res = minimize_scalar(lambda x: -fidelity(10 ** x, N, L, **kw),
                          bounds=(lo, hi), method='bounded', options={'xatol': 1e-7})
    return 10 ** res.x, -res.fun, sig, F


s_opt, F_opt, sig_g, F_g = optimize_sigma(4, L)
print(f"N=4, L={L:g}:  σ_max={s_opt:.5f}  F_max={F_opt:.6f}  (máximo da grade grossa: {F_g.max():.6f})")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogx(sig_g, F_g, '.-', lw=0.8, ms=3, label='grade grossa')
ax.axvline(s_opt, color='k', ls=':', lw=0.8)
ax.plot([s_opt], [F_opt], 'r*', ms=10, label=r'$(\sigma_{\max}, F_{\max})$')
ax.set_xlabel(r'$\sigma$'); ax.set_ylabel(r'$F_{\rm avg}(\pi)$'); ax.set_title(f'N = 4, L = {L:g}')
ax.legend(); fig.tight_layout(); plt.show()

---
## 9. Ajustes de lei de potência (Fig. 3(b))

O artigo mostra que, para $N$ grande,

$$
\boxed{\;1-F_{\max}\simeq A\,N^{-p},\qquad \sigma_{\max}\simeq B\,N^{-q}\;}
$$

Tomando logaritmo, $\ln y=\ln C-p\ln N$ é uma reta. `fit_power_law` faz uma regressão linear
(`np.polyfit` de grau 1) de $\ln y$ contra $\ln N$, só para $N$ no intervalo `fit_range = (4, 20)`
(evita o regime de $N$ pequeno, onde a lei de potência ainda não vale), e devolve $C=e^{\text{intercepto}}$ e
$p=-\text{inclinação}$.

Valores do artigo ($L=0$, $\chi\to\infty$):

$$
1-F_{\max}\approx0.537\,N^{-1.61},\qquad \sigma_{\max}\approx0.350\,N^{-0.81}.
$$

Na figura, os ajustes aparecem como as retas tracejadas (azul: $\sigma_{\max}$; laranja: $1-F$).

In [ ]:
def fit_power_law(N, y, rng=(4, 20)):
    """Ajuste linear em log-log: y = C N^{-p}. Devolve (C, p)."""
    m = (N >= rng[0]) & (N <= rng[1])
    slope, icpt = np.polyfit(np.log(N[m]), np.log(y[m]), 1)
    return np.exp(icpt), -slope


DATA_DIR = 'data'
arquivos = {L_: os.path.join(DATA_DIR, f'fig3_L{L_:g}_chi1000_gamma1_omega0.npz') for L_ in (0, 0.2)}

for L_, f in arquivos.items():
    d = np.load(f)
    A_, p_ = fit_power_law(d['N'], 1 - d['F_max'])
    B_, q_ = fit_power_law(d['N'], d['sigma_max'])
    print(f"L={L_:g}:  1-F = {A_:.3f} N^(-{p_:.3f}),   σ_max = {B_:.3f} N^(-{q_:.3f})")

Com $L=0$ os expoentes reproduzem os do artigo ($p\approx1.61$, $q\approx0.81$), apesar de $\chi=1000$ em vez
de $\infty$. Com $L\neq0$ a largura ótima quase não muda ($q\approx0.81$), mas a infidelidade cai **bem mais
devagar** com $N$: as fases $e^{i\omega L}$ fazem os termos $j$ da soma da seção 3.2 interferirem de forma
menos construtiva.

---
## 10. Verificação: recalcular e comparar com os dados salvos

Os dados do gráfico vêm de `calcular_infidelidade.py`, salvos em `data/fig3_L{L}_chi{χ}_gamma{γ}_omega{ω0}.npz`,
com os campos

| Campo | Conteúdo |
|---|---|
| `N`, `sigma_max`, `F_max` | Fig. 3(b): resultado de `optimize_sigma` para cada $N$ |
| `N_fig3a`, `sigma_fig3a`, `F_curves` | Fig. 3(a): $F_{\rm avg}(\pi)$ na grade fina, uma linha por $N$ |
| `L`, `chi`, `gamma`, `omega0`, ... | parâmetros usados no cálculo |

As funções deste notebook devem reproduzir os valores salvos. Refazemos alguns pontos (a varredura
completa até $N=50$ é lenta).

In [ ]:
d = np.load(arquivos[0.2])
print("L do arquivo:", float(d['L']), "  N de", d['N'].min(), "a", d['N'].max())

print(f"\n{'N':>3} {'σ':>8} {'F salvo':>12} {'F recalc.':>12} {'|dif|':>9}   (Fig. 3a)")
for k, N in enumerate(d['N_fig3a'][:5]):
    for idx in (100, 200):
        s = d['sigma_fig3a'][idx]
        Fr = fidelity(s, int(N), float(d['L']))
        print(f"{N:3d} {s:8.4f} {d['F_curves'][k, idx]:12.8f} {Fr:12.8f} {abs(Fr - d['F_curves'][k, idx]):9.1e}")

print(f"\n{'N':>3} {'σ_max salvo':>12} {'recalc.':>10} {'F_max salvo':>12} {'recalc.':>10}   (Fig. 3b)")
for N in (2, 4):
    i = list(d['N']).index(N)
    s_r, F_r, *_ = optimize_sigma(N, float(d['L']))
    print(f"{N:3d} {d['sigma_max'][i]:12.6f} {s_r:10.6f} {d['F_max'][i]:12.8f} {F_r:10.8f}")

---
## 11. Checagem analítica: limite $\sigma\to0$

Para $\sigma\ll\gamma$ o pacote é estreito em frequência; todos os fatores lentos podem ser avaliados em
$\omega_c$. Então $A=B=a(\omega_c)^2$, a soma em $j$ vale $N\,a(\omega_c)^{2(N-1)}$, e com
$S_1^*(\omega_c)^2=e^{-2i(N-1)\omega_cL}\,t^*(\omega_c)^{2N}$ todas as fases se cancelam exceto
$t^*(\omega_c)^2/\Gamma(\omega_c)^4=1/|\Gamma(\omega_c)|^4$. O que sobra é a integral das quatro gaussianas
com a delta, $\int\!\xi^4=2\sqrt{\pi}\,\sigma$:

$$
\boxed{\;G(\sigma\to0)\simeq 2\sqrt{\pi}\,\sigma\;N\;\left(-\frac{i\gamma^{2}}{\pi}\right)
\frac{\chi K(2\omega_c)}{|\Gamma(\omega_c)|^{4}}\;}
$$

Consequências, visíveis à esquerda da Fig. 3(a):

1. $G\propto\sigma\to0$, logo $\mathrm{Ov}\to1$ e $F_{\rm avg}(\pi)\to0.4$ (pacotes longos não acumulam fase
   condicional);
2. $G$ cresce **linearmente com $N$**: cada sítio adiciona a mesma fase pequena; por isso as curvas de $N$ maior
   sobem mais cedo (em $\sigma$ menor), o que dá $\sigma_{\max}$ decrescente com $N$;
3. o limite **não depende de $L$**.

A expansão vale para $\sigma\ll\gamma/N$ (a mesma escala $\gamma/(2N)$ da fase $t^N$, seção 6): o erro relativo
cresce como $N\sigma/\gamma$, como mostra a tabela abaixo.

In [ ]:
def G_sigma_zero(sigma, N):
    return 2 * np.sqrt(np.pi) * sigma * N * (-1j * gamma ** 2 / np.pi) * chiK(2 * omega_c) / np.abs(Gamma(omega_c)) ** 4


for N in (1, 4, 20):
    for s in (1e-3, 1e-2):
        Gn, Ga = compute_G(s, N, L), G_sigma_zero(s, N)
        print(f"N={N:2d} σ={s:g}:  G_num={Gn:.3e}  G_assint={Ga:.3e}  erro rel.={abs(Gn - Ga) / abs(Ga):.1e}")

---
## 12. O gráfico, elemento por elemento

A figura (estilo da Fig. 3 do artigo, feita em Mathematica) é desenhada por `plot_fig3` em
`plot_infidelidade.py`, a partir do `.npz`.

**Painel (a)** — $F_{\rm avg}(\pi)$ vs $\sigma$:

| Elemento | Código | Expressão / significado |
|---|---|---|
| Curvas | `ax_a.semilogx(sigma_fig3a, F_curves[k])` | $F_{\rm avg}(\pi;\sigma,N)$ (seção 5), uma por $N\in$ `N_fig3a` |
| Eixo $x$ | `xlabel_a`, log, $[10^{-2},1]$ | largura espectral $\sigma$ do pacote (seção 2.5), em unidades de $\gamma$ |
| Eixo $y$ | `ylabel_a`, $[0.35,1]$ | $F_{\rm avg}(\pi)$; o piso sem interação é $0.4$ (seção 11), o teto ideal é $1$ |
| Rótulos "N = k" | `label_y`, `label_shift` | altura fixa por $N$; $x$ = ponto onde a curva cruza essa altura na subida, dividido por `label_shift` |
| Cores | `MMA` | paleta padrão do Mathematica (`ColorData[97]`), na ordem do artigo |

**Painel (b)** — $1-F_{\max}$ e $\sigma_{\max}$ vs $N$ (log-log), para $N\ge$ `N_fig3b_min` $=2$:

| Elemento | Código | Expressão / significado |
|---|---|---|
| Círculos cheios (azul) | `sigma_max` | $\sigma_{\max}(N)$ (seção 8) |
| Círculos vazados (laranja) | `1 - F_max` | infidelidade mínima $1-F_{\max}(N)$ (seção 8) |
| Tracejados | `fit_power_law` | $B N^{-q}$ e $A N^{-p}$, ajustados em $N\in[4,20]$ (seção 9) |
| Ticks do eixo $y$ | lista fixa filtrada por `lo`, `hi` | só os valores dentro da faixa dos dados |

**Estilo geral**: ticks para dentro nos quatro lados, números sem serifa (`_tick_font`), rótulos em serifa com
`mathtext.fontset='cm'` (Computer Modern), e `svg.fonttype='path'` (texto vira curva; o SVG não depende de
fontes instaladas). O arquivo de saída é `fig3_L{L}_Nmax{N_max}.svg`.

In [ ]:
import io

import plot_infidelidade as pi_

r = dict(np.load(pi_.arquivo(0.2)))
fig = pi_.plot_fig3(r, outfile=io.BytesIO())   # mesma função do script (salva em memória, não sobrescreve o .svg)
plt.show()

**Leitura física do resultado.**

* $\sigma\to0$: $F_{\rm avg}(\pi)\to0.4$ para todo $N$ e $L$ (seção 11).
* Existe um $\sigma$ ótimo: pacotes estreitos em frequência acumulam pouca fase condicional, pacotes largos
  (curtos no tempo) são deformados e vazam para fora do subespaço computacional.
* Aumentar $N$ divide a interação entre vários sítios fracos: o pico sobe e se desloca para $\sigma$ menor
  ($\sigma_{\max}\propto N^{-0.81}$) e a infidelidade cai como lei de potência.
* Com $L\neq0$ as fases de propagação entre sítios estragam a interferência entre os termos $j$, e a queda da
  infidelidade com $N$ fica muito mais lenta do que no artigo ($L=0$).